# Chapter 3: Data Storage and Management

This notebook complements Chapter 3 by turning storage design into an asset-classification exercise.

Use the retail ranking track for orders, clickstream, feature tables, benchmarks, and embeddings. Use the support-assistant track for documents, chunks, prompt templates, context traces, and retrieval artifacts.

The chapter makes one additional point explicit: LLM systems need storage for both **knowledge** and **behavior**. Knowledge lives in documents, chunks, embeddings, and indexes. Behavior lives partly in prompts, routing rules, system instructions, tool permissions, and traces.


In [ ]:
from pathlib import Path

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")


def ensure_path_exists(path: Path, kind: str) -> Path:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing expected {kind}: {path}. "
            "Make sure you are running this notebook from inside the companion repository."
        )
    return path

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
print("Companion root located successfully.")

In [ ]:
from importlib.util import module_from_spec, spec_from_file_location

import pandas as pd

module_path = ensure_path_exists(
    COMPANION_ROOT / "code" / "storage" / "storage_layers.py",
    "helper module",
)
spec = spec_from_file_location("storage_layers", module_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load helper module spec from {module_path}")
storage_layers = module_from_spec(spec)
spec.loader.exec_module(storage_layers)

load_storage_inventory = storage_layers.load_storage_inventory
summarize_storage_layers = storage_layers.summarize_storage_layers
build_retention_plan = storage_layers.build_retention_plan
build_vector_lineage = storage_layers.build_vector_lineage
recommend_storage_layer = storage_layers.recommend_storage_layer

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 150)

### Why the Helper Setup Exists

The setup code loads a small storage helper so the notebook can stay focused on judgment instead of boilerplate. The chapter's real question is not how to parse a CSV. It is how to classify assets by authority, retention, lineage strength, and failure mode.

As you work through the outputs, treat the helper functions as stand-ins for the platform's own catalog and storage-planning logic.

## Part 1: Inspect the Storage Inventory

The inventory now works as a storage-asset taxonomy. Do not only ask where an asset fits technically. Ask what metadata it needs and what failure appears first if its lineage is weak.

In [ ]:
inventory_path = ensure_path_exists(
    COMPANION_ROOT / "datasets" / "storage" / "sample_storage_inventory.csv",
    "dataset",
)
storage_df = load_storage_inventory(inventory_path)
storage_df[[
    "asset_name",
    "storage_layer",
    "workload_type",
    "access_pattern",
    "has_embeddings",
]].sort_values("asset_name").reset_index(drop=True)

### What to Notice in the Inventory Output

The inventory table should immediately tell you whether the platform is separating authoritative assets from derived serving assets. If documents, benchmarks, embeddings, and prompt-related artifacts all look like undifferentiated files, the storage layer is already hiding important engineering contracts.


## Part 1A: Surface Prompt and Context Storage

Chapter 3 is strongest when prompt and context assets appear in the same inventory as orders, features, and vectors. The rows below make that explicit.

In [ ]:
prompt_context_assets = storage_df.loc[
    storage_df['asset_name'].isin([
        'prompt_template_registry',
        'system_instruction_versions',
        'context_trace_log',
        'tool_schema_registry',
        'rag_benchmark_set',
    ]),
    ['asset_name', 'storage_layer', 'workload_type', 'retention_days', 'lineage_parent'],
].sort_values('asset_name').reset_index(drop=True)
prompt_context_assets

### Why These Rows Matter

Once prompt registries, context traces, tool schemas, and RAG benchmarks are visible here, the storage lesson becomes sharper: LLM systems store behavior and evidence, not only knowledge and vectors.

## Part 2: Summarize the Platform by Layer

This view shows how much of the platform lives in each storage layer and how many assets in each layer carry embeddings.


In [ ]:
layer_summary = summarize_storage_layers(storage_df)
layer_summary


### Interpreting the Layer Summary

Look for concentration, not just counts. A storage layer with many assets and many embedding-bearing artifacts is often where reproducibility and access-control pressure accumulate fastest. That is usually a signal to tighten metadata and lineage before adding more platform surface area.


## Part 3: Surface Evaluation and Benchmark Assets

Benchmarks and relevance judgments belong in the same storage conversation as features and vectors. If they drift silently or are overwritten, model comparison becomes less trustworthy even when the training code stays stable.

In [ ]:
benchmark_assets = storage_df.loc[
    storage_df["workload_type"] == "benchmark",
    [
        "asset_name",
        "storage_layer",
        "file_format",
        "retention_days",
        "lineage_parent",
    ],
].sort_values("asset_name").reset_index(drop=True)
benchmark_assets


## Part 4: Plan Retention and Tiering

Retention is not only about cost. It is also about what must remain available for replay, audit, or comparison later.


In [ ]:
retention_plan = build_retention_plan(storage_df)
retention_plan


### Interpreting the Retention Plan

The retention output matters when it changes what must remain reconstructable. If an asset is marked for archive-with-versioning, that is the notebook's way of saying the data may be cold operationally but still hot for audits, replay, or benchmark comparison.


## Part 5: Track Embedding Lineage

A vector index is not enough by itself. The platform still needs source lineage, embedding model version, and a clear relationship to the underlying asset. The same storage logic applies to prompt registries: a prompt template should carry version, owner, intended model, tests, and rollback state.

In [ ]:
vector_lineage = build_vector_lineage(storage_df)
vector_lineage


### Interpreting the Lineage View

Use the lineage view to check whether vector-serving state can be traced back to a source snapshot and embedding model. If that chain is weak, the platform can return fast answers while still being unable to explain which knowledge state produced them.


## Part 5A: Stress the Retrieval Asset Lineage Model

A vector index that answers quickly but cannot be traced back to a corpus snapshot is serving-only state with weak evidence behind it.

In [ ]:
vector_without_lineage = pd.DataFrame([
    {
        'asset_name': 'support_index_live',
        'storage_layer': 'vector_index',
        'embedding_model': '',
        'lineage_parent': '',
        'snapshot_date': 'unknown',
        'why_dangerous': 'fast retrieval with no reconstructable corpus or chunking provenance',
    }
])
vector_without_lineage

### Why a Vector Index Without Lineage Is Dangerous

The index may still serve low-latency answers, but the team cannot explain which corpus snapshot, chunking policy, or embedding model produced the live behavior. That is exactly why vector state should not be treated as source of truth.

## Part 6: Make Quick Storage Decisions

Use these quick cases to test a stronger distinction:

- what is authoritative storage?
- what is derived but still needs durable lineage?
- what can move cold without damaging reproducibility?
- what should never be treated as source-of-truth state, even if it is fast to query?

In [ ]:
cases = [
    {
        "name": "benchmark_corpus",
        "access_pattern": "analytics",
        "latency_class": "medium",
        "data_shape": "tabular",
        "mutability": "snapshot",
    },
    {
        "name": "retrieval_serving_index",
        "access_pattern": "semantic_search",
        "latency_class": "low",
        "data_shape": "vector",
        "mutability": "snapshot",
    },
]

decision_rows = []
for case in cases:
    recommendation = recommend_storage_layer(
        access_pattern=case["access_pattern"],
        latency_class=case["latency_class"],
        data_shape=case["data_shape"],
        mutability=case["mutability"],
    )
    decision_rows.append({**case, **recommendation})

pd.DataFrame(decision_rows)


### How to Read the Storage Recommendations

Read the final recommendations as a storage-contract review, not a product feature list. For each case, ask which layer is authoritative, which layer is derived for speed or convenience, which metadata makes reconstruction possible later, and which fast-serving structure should never be mistaken for source-of-truth storage.


## Part 6A: Map Assets to Storage Responsibility

The final storage decision should say more than where an asset lives. It should say whether the asset is authoritative, derived, reconstructable, serving-only, or evidence-only.

In [ ]:
def storage_role(row):
    if row['workload_type'] in {'transactions', 'master_data'}:
        return 'authoritative'
    if row['workload_type'] in {'benchmark', 'evidence'}:
        return 'evidence-only'
    if row['storage_layer'] == 'vector_index' or row['access_pattern'] == 'serving':
        return 'serving-only'
    if row['workload_type'] in {'behavior', 'metadata'}:
        return 'behavioral or control state'
    return 'derived analytical state'

asset_responsibility_map = storage_df[[
    'asset_name',
    'storage_layer',
    'workload_type',
    'lineage_parent',
    'retention_days',
]].copy()
asset_responsibility_map['asset_role'] = storage_df.apply(storage_role, axis=1)
asset_responsibility_map['reconstructable'] = storage_df['lineage_parent'].fillna('').ne('') | storage_df['workload_type'].isin(['transactions', 'benchmark', 'behavior'])
asset_responsibility_map['source_of_truth_safety'] = asset_responsibility_map['asset_role'].map({
    'authoritative': 'safe as source of truth',
    'derived analytical state': 'derived but reconstructable',
    'behavioral or control state': 'behavioral state, not business truth',
    'serving-only': 'unsafe as source of truth',
    'evidence-only': 'evidence-only',
})
asset_responsibility_map['first_failure_mode'] = asset_responsibility_map['asset_role'].map({
    'authoritative': 'cannot reconstruct truth',
    'derived analytical state': 'training or evaluation cannot be rebuilt cleanly',
    'behavioral or control state': 'generated behavior cannot be explained later',
    'serving-only': 'fast but stale or weakly explained behavior',
    'evidence-only': 'release comparisons lose fairness or audit value',
})
asset_responsibility_map.sort_values(['asset_role', 'asset_name']).reset_index(drop=True)

### Retention Scenario

A common support-assistant question is whether raw prompt traces can be deleted after 30 days while redacted evaluation evidence remains longer. The answer is often yes, but only if the retained evaluation record still preserves prompt version, corpus snapshot, failure category, and enough trace structure to support benchmark comparison without keeping the full sensitive payload.

## Part 6B: Inspect Storage Artifacts

The chapter points readers to compact storage artifacts because storage strategy should be reviewable outside the notebook. Inspect the sample inventory and lineage record below as examples of authoritative-versus-derived storage reasoning.

In [ ]:
inventory_artifact_path = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "storage" / "storage_inventory_sample.csv",
    "artifact",
)
lineage_artifact_path = ensure_path_exists(
    COMPANION_ROOT / "artifacts" / "storage" / "embedding_lineage.yaml",
    "artifact",
)

print("storage_inventory_sample.csv")
print(pd.read_csv(inventory_artifact_path).head())
print("\nembedding_lineage.yaml")
print("-" * 60)
print("\n".join(lineage_artifact_path.read_text(encoding="utf-8").splitlines()[:18]))
print("\nstorage_layers.py")
print("-" * 60)
print("\n".join(module_path.read_text(encoding="utf-8").splitlines()[:18]))

## Exercise

### Concept check
- Classify the major assets in this notebook as authoritative, derived, serving-only, behavioral or control, or evidence-only.

### Diagnostic scenario
- Explain why `support_index_live` is not a source of truth when corpus snapshot, chunking configuration, embedding model, and build timestamp are missing.

### Artifact-building exercise
- Design a retention compromise for raw prompt traces and longer-lived evaluation evidence. Use columns for keep or delete, redact or not, retention period, and reason.

### Notebook extension
- Add one new asset row to the responsibility map, classify its storage role, and explain the first failure mode if its lineage becomes weak.